In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone https://github.com/Zheng-Chong/CatVTON.git

In [ ]:
%cd /content/CatVTON_3DGS
!pip uninstall -y torch torchvision torchaudio xformers torchao
!pip install -r requirements.txt

In [ ]:
%cd /content/CatVTON

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
repo_path = snapshot_download(repo_id="zhengchong/CatVTON")

automasker = AutoMasker(
    densepose_ckpt=os.path.join(repo_path, "DensePose"),
    schp_ckpt=os.path.join(repo_path, "SCHP"),
    device=device
)

In [ ]:
def show_try_on_res(person, mask, try_on_result):
    fig, axs = plt.subplots(1, 3, figsize=(15, 6))

    axs[0].imshow(person)
    axs[0].set_title("Front Result (Reference)")
    axs[0].axis('off')

    axs[1].imshow(mask)
    axs[1].set_title("Try-On Mask")
    axs[1].axis('off')

    axs[2].imshow(try_on_result.resize(target_size))
    axs[2].set_title("Try-On Result")
    axs[2].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
from PIL import Image
from pathlib import Path

def remove_background_from_all_in_dir(target_dir: Path, automasker):
    for file_path in target_dir.iterdir():
        if file_path.is_file() and file_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
            img_pil = Image.open(file_path).convert("RGB")

            # Use automasker to get human parsing
            res = automasker(img_pil)

            # schp_atr contains class 0 for background, >0 for human/clothes/etc.
            schp_atr = np.array(res['schp_atr'])

            # Create a binary mask (255 for human, 0 for background)
            mask = (schp_atr > 0).astype(np.uint8) * 255

            # Optional: Smooth the mask slightly to avoid harsh jagged edges
            kernel = np.ones((3, 3), np.uint8)
            mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

            # Convert img to cv2 format (RGB -> BGR for saving)
            img_cv2 = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

            # Apply mask (sets background to pure black)
            img_bg_removed = cv2.bitwise_and(img_cv2, img_cv2, mask=mask)

            # Overwrite the original image in the folder so 3DGS uses the masked version
            cv2.imwrite(str(file_path), img_bg_removed)

In [ ]:
from tqdm import tqdm

def calc_body_views(person_folder, automasker):
    # Get all image files in person_folder excluding front_person_path
    all_files = [
        os.path.join(person_folder, f)
        for f in os.listdir(person_folder)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]
    body_params = {}
    for file_path in tqdm(all_files):
        person_img = Image.open(file_path).convert("RGB")

        automasker_res = automasker(person_img)
        densepose_img = np.array(automasker_res['densepose'])

        body_params[file_path] = {'view_type': determine_view_angle(densepose_img)}
    return body_params

### Detect True Front using YOLOv8n-pose + Depth Anything V2
Instead of pixel width (which is vulnerable to camera distance), we calculate the Z-depth of both shoulders. The image where `abs(left_shoulder_depth - right_shoulder_depth)` is closest to 0 represents the most perfectly aligned front view.

In [ ]:
import cv2
import math
import numpy as np
from ultralytics import YOLO

# 1. Initialize YOLOv8 Pose Model (using nano for speed, can be changed to yolov8x-pose.pt for highest accuracy)
yolo_pose_predictor = YOLO('yolov8n-pose.pt')

# 2. Function to extract shoulder width and keypoints
def get_yolov8n_shoulder_width_and_kpts(img_pil):
    # Convert PIL RGB to OpenCV BGR
    img_bgr = np.array(img_pil)[:, :, ::-1]

    # Run YOLOv8 pose inference
    results = yolo_pose_predictor(img_bgr, verbose=False)

    if len(results) == 0 or results[0].keypoints is None or len(results[0].keypoints.data) == 0:
        return None, np.zeros((17, 3))

    # Get keypoints for the first detected person
    # YOLOv8 format: [num_persons, 17, 3] where 3 is (x, y, confidence)
    keypoints = results[0].keypoints.data[0].cpu().numpy()

    # COCO format: 5 is Left Shoulder, 6 is Right Shoulder
    l_shoulder = keypoints[5]
    r_shoulder = keypoints[6]

    # YOLOv8 confidence scores are normalized 0-1, so 0.3 is a reasonable threshold
    if l_shoulder[2] > 0.3 and r_shoulder[2] > 0.3:
        # 2D Euclidean distance (accounts for tilt/lean)
        width = math.hypot(l_shoulder[0] - r_shoulder[0], l_shoulder[1] - r_shoulder[1])
        return width, keypoints
    return None, keypoints


In [ ]:
def get_shoulder_depth_diff(img_pil, keypoints):
    # keypoints format: [17, 3] from Detectron2 (x, y, score)
    # 5 is Left Shoulder, 6 is Right Shoulder
    l_shoulder = keypoints[5]
    r_shoulder = keypoints[6]

    # Prepare image for depth model
    inputs = depth_image_processor(images=img_pil, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = depth_model(**inputs)
        predicted_depth = outputs.predicted_depth

    # Interpolate to original size
    prediction = torch.nn.functional.interpolate(
        predicted_depth.unsqueeze(1),
        size=img_pil.size[::-1],
        mode="bicubic",
        align_corners=False,
    )

    depth_map = prediction.squeeze().cpu().numpy()

    # Get depth at shoulder coordinates (y, x)
    # Note: image coordinates are (x, y), numpy array is [y, x]
    l_x, l_y = int(l_shoulder[0]), int(l_shoulder[1])
    r_x, r_y = int(r_shoulder[0]), int(r_shoulder[1])

    # Ensure coordinates are within bounds
    h, w = depth_map.shape
    l_x, l_y = np.clip(l_x, 0, w-1), np.clip(l_y, 0, h-1)
    r_x, r_y = np.clip(r_x, 0, w-1), np.clip(r_y, 0, h-1)

    l_depth = depth_map[l_y, l_x]
    r_depth = depth_map[r_y, r_x]

    return abs(l_depth - r_depth)

### Detect True Front using MediaPipe 3D Pose Landmarks
Instead of a 2D pose model + depth estimation model, MediaPipe provides a `z` coordinate directly out-of-the-box which estimates the landmark depth relative to the subject's center of mass. We can find the image where `abs(left_shoulder.z - right_shoulder.z)` is minimized.

In [ ]:
import cv2
import math
import numpy as np
import matplotlib.pyplot as plt
import os
import urllib.request
from PIL import Image
import sys

# Download the model asset if it doesn't exist
model_path = 'pose_landmarker_heavy.task'
if not os.path.exists(model_path):
    urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task', model_path)

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Initialize MediaPipe Pose Landmarker
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=False,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5)
mp_detector = vision.PoseLandmarker.create_from_options(options)

def get_mediapipe_shoulder_info(img_pil):
    # Convert PIL to NumPy array (MediaPipe expects RGB, which PIL provides)
    img_np = np.array(img_pil)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_np)

    detection_result = mp_detector.detect(mp_image)

    if not detection_result.pose_landmarks:
        return None, None, None

    landmarks = detection_result.pose_landmarks[0]
    # MediaPipe Pose: 11 is Left Shoulder, 12 is Right Shoulder
    l_shoulder = landmarks[11]
    r_shoulder = landmarks[12]

    # Visibility check (0.0 to 1.0)
    if l_shoulder.visibility < 0.5 or r_shoulder.visibility < 0.5:
        return None, None, None

    # x, y are normalized [0.0, 1.0]. Multiply by dimensions for pixel coords
    h, w, _ = img_np.shape
    l_x, l_y = l_shoulder.x * w, l_shoulder.y * h
    r_x, r_y = r_shoulder.x * w, r_shoulder.y * h

    # 2D Width in pixels
    width = math.hypot(l_x - r_x, l_y - r_y)

    # Calculate depth difference using MediaPipe's intrinsic z coordinate
    # Z is roughly the depth in 'image scale' relative to the hips
    depth_diff = abs(l_shoulder.z - r_shoulder.z)

    coords = {
        'left': (l_x, l_y),
        'right': (r_x, r_y)
    }
    return width, depth_diff, coords


def calc_mediapipe_shoulder_depth(body_params, view_type='front'):
    shoulder_depth_diffs = {}

    # Initialize max_width on the fly
    max_width = 0.0

    for file_path in body_params.keys():
        # Use the initial mask filter to process only rough front candidates
        if body_params[file_path]['view_type'] == view_type:
            img_pil = Image.open(file_path).convert("RGB")
            width, depth_diff, coords = get_mediapipe_shoulder_info(img_pil)

            if width is not None:
                # Update max_width dynamically
                max_width = max(max_width, width)

                # Optional width filter to ignore obvious false positives
                if width < (max_width * 0.6):
                    continue

                shoulder_depth_diffs[file_path] = (depth_diff, coords, width)

    return shoulder_depth_diffs


In [ ]:
!mkdir /content/clothes

# Modifying the U-Net for Reference-Driven Editing (consistent side-view try-on generation)

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from diffusers.models.attention_processor import Attention

class ReferenceAttentionProcessor:
    def __init__(self):
        self.is_reference_pass = True
        self.bank_k = []
        self.bank_v = []
        self.step_idx = 0
        self.injection_steps = range(5, 45)

    def clear_bank(self):
        self.bank_k.clear()
        self.bank_v.clear()
        self.step_idx = 0

    def reset_read_index(self):
        self.step_idx = 0

    def __call__(
        self,
        attn: Attention,
        hidden_states: torch.Tensor,
        encoder_hidden_states=None,
        attention_mask=None,
        **kwargs,
    ):
        batch_size, sequence_length, _ = hidden_states.shape

        # 1. Get Q, K, V using built-in attention layers
        query = attn.to_q(hidden_states)
        key = attn.to_k(hidden_states)
        value = attn.to_v(hidden_states)

        inner_dim = key.shape[-1]
        head_dim = inner_dim // attn.heads

        # Transform dimensions to 4D for Flash Attention: (batch, heads, seq_len, head_dim)
        query = query.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)
        key = key.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)
        value = value.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)

        if self.is_reference_pass:
            self.bank_k.append(key.clone().detach().cpu())
            self.bank_v.append(value.clone().detach().cpu())
        else:
            if self.step_idx in self.injection_steps and self.step_idx < len(self.bank_k):
                ref_k = self.bank_k[self.step_idx].to(key.device)
                ref_v = self.bank_v[self.step_idx].to(value.device)

                if key.shape[0] != ref_k.shape[0]:
                    multiplier = key.shape[0] // ref_k.shape[0]
                    ref_k = ref_k.repeat(multiplier, 1, 1, 1)
                    ref_v = ref_v.repeat(multiplier, 1, 1, 1)

                key = torch.cat([ref_k, key], dim=2)
                value = torch.cat([ref_v, value], dim=2)

            self.step_idx += 1

        # 3. Memory-efficient Attention computation (SDPA)
        # Using 4D tensors enables memory-efficient / Flash Attention in PyTorch
        with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=False, enable_mem_efficient=True):
            try:
                hidden_states = F.scaled_dot_product_attention(
                    query, key, value, dropout_p=0.0, is_causal=False
                )
            except RuntimeError:
                # Fallback to math if necessary
                with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=True, enable_mem_efficient=True):
                    hidden_states = F.scaled_dot_product_attention(
                        query, key, value, dropout_p=0.0, is_causal=False
                    )

        # Reshape back to expected format
        hidden_states = hidden_states.transpose(1, 2).reshape(batch_size, -1, attn.heads * head_dim)

        # Linear projections and dropout
        hidden_states = attn.to_out[0](hidden_states)
        if len(attn.to_out) > 1:
            hidden_states = attn.to_out[1](hidden_states)

        return hidden_states

# --- Integration into CatVTON pipeline ---
# 2. Replace standard attention layers in U-Net with our custom one ONLY for up_blocks!
attn_processors = {}
for name in pipeline.unet.attn_processors.keys():
    # Custom processor only for attn1 in up_blocks
    if "attn1" in name and "up_blocks" in name:
        print(f"replaced attn_processors[{name}] = {pipeline.unet.attn_processors.get(name, None)} with custom attention processor")
        attn_processors[name] = ReferenceAttentionProcessor()
    else:
        attn_processors[name] = pipeline.unet.attn_processors[name]

pipeline.unet.set_attn_processor(attn_processors)


## Install camenduru 3DGS (3DGS version for colab)

In [ ]:
%cd /content/gaussian-splatting

In [ ]:
!{sys.executable} -m pip install -q /content/gaussian-splatting/submodules/diff-gaussian-rasterization

In [ ]:
!sed -i '1i #include <float.h>' /content/gaussian-splatting/submodules/simple-knn/simple_knn.cu

%cd /content/gaussian-splatting
!pip install ./submodules/simple-knn

In [ ]:
!mkdir -p /content/gaussian-splatting/input/vadim
# !cp /content/drive/MyDrive/weavella/images/Photos_Manual_Selection/9/vadim.MOV /content/gaussian-splatting/input/vadim/
!gdown 1MPlCmym-gT7Gndj92rsbMxIJMJF-rQhX -O /content/gaussian-splatting/input/vadim/vadim.MOV

## Init front/back cloth images

In [ ]:
# different front-back
garment_folder = "/content/clothes/cloth_from_zoolando_website/jack-and-jones-jorsnake-crew-neck-t-shirt-z-nadrukiem-antique-white-ja222o5zp-a11"
# pure flat-out t-shirt image
front_garment_path = garment_folder + "/c238569c66eb42578c5a787c54e43731.webp"
back_garment_path  = garment_folder + "/ceac3aa9af9b451d8aa29df6cf5db6dc.webp"

front_garment_image = Image.open(front_garment_path).convert("RGB").resize(target_size)
back_garment_image  = Image.open(back_garment_path).convert("RGB").resize(target_size)

In [ ]:
# different front-back + side image
garment_folder = "/content/clothes/cloth_from_zoolando_website/the-north-face-retro-earth-short-sleeve-graphic-t-shirt-z-nadrukiem-white-dune-th322o0e8-a11"

# front_garment_path = garment_folder + "/e8cb036024b94af8aaeb5b8d38dfcb7c.webp" # pure flat-out t-shirt image
front_garment_path = garment_folder + "/e3c8055f31754cbbbeff96938543fdc1.webp" # on person
# back_garment_path  = garment_folder + "/95c143c6284549e99409498bf1b92679.webp" # pure flat-out t-shirt image
back_garment_path  = garment_folder + "/0c220ec6788a456194fa5184802d1743.webp" # on person
side_garment_path  = garment_folder + "/c172d2683ba54fe299fdbc6b5023658d.webp" # on person

front_garment_image = Image.open(front_garment_path).convert("RGB").resize(target_size)
back_garment_image  = Image.open(back_garment_path).convert("RGB").resize(target_size)
side_garment_image  = Image.open(side_garment_path).convert("RGB").resize(target_size)

In [ ]:
person_folder = "/content/gaussian-splatting/input/vadim/init_frames"
body_params = calc_body_views(person_folder, automasker)

In [ ]:
# Sort and display (closest to 0 is better)
sorted_shoulder_diffs = dict(sorted(calc_yolov8n_deepanything_shoulder_depth(body_params).items(), key=lambda item: item[1]))
print("\nShoulder depth differences (sorted):")
for name, d in sorted_shoulder_diffs.items():
    print(f"{name}: {d:.4f}")

print(f"\nTrue front image detected (Depth Method): {list(sorted_shoulder_diffs.keys())[0]} with depth difference {list(sorted_shoulder_diffs.values())[0]:.4f}")

### Mediapipe results visualization

In [ ]:
# Let's visualize the top 5 images with the lowest shoulder depth difference (MediaPipe)
mp_top_images = list(mp_sorted_shoulder_diffs.keys())[:5]

fig, axes = plt.subplots(1, len(mp_top_images), figsize=(20, 5))
if len(mp_top_images) == 1:
    axes = [axes]

for ax, img_name in zip(axes, mp_top_images):
    img_pil = Image.open(img_name).convert("RGB")

    depth_diff, coords, width = mp_sorted_shoulder_diffs.get(img_name)

    ax.imshow(img_pil)

    if coords is not None:
        l_x, l_y = coords['left']
        r_x, r_y = coords['right']

        # Use blue/cyan to distinguish from the YOLOv8 red/yellow visualization
        ax.scatter([l_x, r_x], [l_y, r_y], c='blue', s=40, marker='o')
        ax.plot([l_x, r_x], [l_y, r_y], c='cyan', linestyle='--', linewidth=2)
        ax.set_title(f"{img_name}\nMP Width: {width:.1f}px\nMP Z-Diff: {depth_diff:.4f}")
    else:
        ax.set_title(f"{img_name}\n(Not detected)")

    ax.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
def select_img_with_smallest_shoulder_depth_diff(body_params, view_type):
    depth_diff = calc_mediapipe_shoulder_depth(body_params, view_type)
    return sorted(depth_diff.items(), key=lambda item: item[1][0])[0][0]

## Custom mask generators

In [ ]:
from transformers import SegformerImageProcessor, AutoModelForSemanticSegmentation
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

processor = SegformerImageProcessor.from_pretrained("mattmdjaga/segformer_b2_clothes")
model = AutoModelForSemanticSegmentation.from_pretrained("mattmdjaga/segformer_b2_clothes").to(device)

def segformer_refine_mask(coarse_result, init_automasker_res=None, dilate_size=1, dilate_iter=5, close_ksize=35):
    # --- Phase 2: Fine Mask Refinement (SegFormer) ---
    print("Phase 2: Refining Mask from Coarse Result using SegFormer...")

    # Run SegFormer on the coarse result image
    inputs = processor(images=coarse_result, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits.cpu()
    upsampled_logits = nn.functional.interpolate(
        logits,
        size=coarse_result.size[::-1], # (W, H) -> (H, W)
        mode="bilinear",
        align_corners=False,
    )
    pred_seg = upsampled_logits.argmax(dim=1)[0].numpy()

    # Segformer B2 Clothes Upper-clothes class is 4
    upper_clothes_mask = (pred_seg == 4).astype(np.uint8) * 255

    # --- Combine with Initial Coarse Mask ---
    if init_automasker_res is not None:
        print("  -> Combining with initial coarse mask...")
        initial_upper_clothes_mask = extract_upper_clothes_mask(init_automasker_res)
        upper_clothes_mask = cv2.bitwise_or(upper_clothes_mask, initial_upper_clothes_mask)

    # --- Fix Holes and Small Artifacts ---
    # 1. Morphological Opening (removes small white artifacts/noise)
    kernel_small = np.ones((5, 5), np.uint8)
    cleaned_mask = cv2.morphologyEx(upper_clothes_mask, cv2.MORPH_OPEN, kernel_small)

    # 2. Fill internal holes using Contours (Less necessary for SegFormer, but good for safety)
    contours, _ = cv2.findContours(cleaned_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled_mask = np.zeros_like(cleaned_mask)
    cv2.drawContours(filled_mask, contours, -1, 255, thickness=cv2.FILLED)

    # 3. Morphological Closing to fill concave "bays"
    if close_ksize > 0:
        kernel_close = np.ones((close_ksize, close_ksize), np.uint8)
        closed_mask = cv2.morphologyEx(filled_mask, cv2.MORPH_CLOSE, kernel_close)
    else:
        closed_mask = filled_mask

    # --- Dilation for blending ---
    kernel = np.ones((dilate_size, dilate_size), np.uint8)
    refined_mask_np = cv2.dilate(closed_mask, kernel, iterations=dilate_iter)

    # --- Restore Intentional Holes (Bags, Belts, Hair, Scarves) ---
    if init_automasker_res is not None:
        schp_atr = np.array(init_automasker_res['schp_atr'])
        preserve_mask = np.isin(schp_atr, [2, 8, 16, 17]).astype(np.uint8) * 255
    else:
        # SegFormer Labels: Hair=2, Belt=8, Bag=16, Scarf=17
        preserve_mask = np.isin(pred_seg, [2, 8, 16, 17]).astype(np.uint8) * 255

    # Subtract the preserved accessories from the clothing mask
    refined_mask_np[preserve_mask == 255] = 0

    refined_mask = Image.fromarray(refined_mask_np).convert("L")

    return refined_mask, [
        Image.fromarray(upper_clothes_mask).convert("L"),
        Image.fromarray(cleaned_mask).convert("L"),
        Image.fromarray(filled_mask).convert("L"),
        Image.fromarray(closed_mask).convert("L"),
        Image.fromarray(refined_mask_np).convert("L")
    ]

In [ ]:
def _set_reference_pass(pipeline, is_ref_pass):
    for proc in pipeline.unet.attn_processors.values():
        if isinstance(proc, ReferenceAttentionProcessor):
            proc.is_reference_pass = is_ref_pass
            proc.clear_bank() if is_ref_pass else proc.reset_read_index()

def paste_back(person_img, tryon_img, garment_mask, feather=11):
    """Composite the garment region of `tryon_img` onto `person_img`, feathering the seam."""
    size = person_img.size
    p = np.array(person_img.convert("RGB")).astype(np.float32)
    t = np.array(tryon_img.convert("RGB").resize(size)).astype(np.float32)
    m = np.array(garment_mask.convert("L").resize(size)).astype(np.float32)
    if feather > 0:
        k = feather | 1                        # force odd kernel
        m = cv2.GaussianBlur(m, (k, k), 0)
    a = (m / 255.0)[..., None]                 # soft alpha in [0, 1]
    out = (t * a + p * (1.0 - a)).clip(0, 255).astype(np.uint8)
    return Image.fromarray(out)


def dilate_mask(mask, px):
    """Grow a binary mask by `px` pixels (ellipse kernel). Used to swallow the thin strip of the
    ORIGINAL garment that SCHP under-segments at the sleeve hem, so it is not left in the composite."""
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (px, px))
    dilated = cv2.dilate(np.array(mask.convert("L")), kernel, iterations=1)
    return Image.fromarray(dilated).convert("L")



def two_phase_tryon(person_img, garment_img, automasker, pipeline, mask_refiner_fun,
                    is_ref_pass=True, coarse_steps=50, fine_steps=50, seed=42, mask_dilate=14,
                    guidance_scale=2.5):
    # --- Phase 1: Coarse try-on with the WIDE agnostic mask (best garment fidelity) ---
    print("Phase 1: Coarse mask + try-on...")
    coarse_mask_result = automasker(person_img)
    coarse_mask = coarse_mask_result['mask']

    _set_reference_pass(pipeline, is_ref_pass)
    generator = torch.Generator(device=device).manual_seed(seed)
    coarse_result = pipeline(
        image=person_img, condition_image=garment_img, mask=coarse_mask,
        num_inference_steps=coarse_steps, guidance_scale=guidance_scale, generator=generator
    )[0]

    # --- Cloth mask = union(new upper-clothes, original upper-clothes), holes filled, then grown ---
    # The dilation swallows the thin strip of the ORIGINAL garment SCHP under-segments at the hem.
    refined_mask, masks_inbetween = mask_refiner_fun(coarse_result, init_automasker_res=coarse_mask_result)
    composite_mask = dilate_mask(refined_mask, mask_dilate)

    # --- Phase 2: Paste Phase-1 garment onto the ORIGINAL person ---
    print("Phase 2: Paste-back composite...")
    composite_result = paste_back(person_img, coarse_result, composite_mask)

    # --- Phase 3: 2nd try-on ON THE COMPOSITE with the cloth mask ---
    # The composite no longer contains the original shirt, so the diffusion harmonizes the seam and
    # refines the garment WITHOUT mixing in the old cloth.
    print("Phase 3: Refine try-on on the composite...")
    _set_reference_pass(pipeline, is_ref_pass)
    generator = torch.Generator(device=device).manual_seed(seed)
    final_result = pipeline(
        image=composite_result, condition_image=garment_img, mask=composite_mask,
        num_inference_steps=fine_steps, guidance_scale=guidance_scale, generator=generator
    )[0]

    return coarse_result, refined_mask, final_result, masks_inbetween + [composite_mask, composite_result]

## Front/Back/Side split Try-on per frame
This takes a lot of time! You can skip this and just download the results archive from Google Drive: [cell 'Download try-on results from Google Drive'](https://colab.research.google.com/github/Eikthyrnir/CatVTON_3DGS/blob/main/CatVTON_3DGS_pipeline.ipynb#scrollTo=GagRKKpNqbmO&line=1&uniqifier=1)

In [ ]:
NUM_INFERENCE_STEPS_CAT_VTON = 50

### side try-on

### front try-on

In [ ]:
for person_path in front_images: # Process every 10th image
    person_img = Image.open(person_path).convert("RGB").resize(target_size)

    # Run the two-phase pipeline with the new SegFormer refinement
    coarse_res, refined_mask, final_res, masks_refinement_steps = two_phase_tryon(
        person_img,
        front_garment_image,
        automasker,
        pipeline,
        schp_refine_mask, # segformer_refine_mask
        is_ref_pass=False
    )

    show_multiple_images((person_img, coarse_res, refined_mask, final_res))
    show_multiple_images(masks_refinement_steps)

    # Save the front result as well to complete the set
    filename = os.path.basename(person_path)
    output_path = os.path.join(output_folder, filename)
    final_res.save(output_path)
    print(f"\nSaved front result to: {output_path}")

In [ ]:
# Generate masks using automasker

print("--- PASS 1: Reference Generation (Back) ---")

# Run the two-phase pipeline with the new SegFormer refinement
coarse_res, refined_mask, final_res, masks_refinement_steps = two_phase_tryon(
    back_person_img,
    back_garment_image,
    automasker,
    pipeline,
    schp_refine_mask, # segformer_refine_mask
    is_ref_pass=True,
    guidance_scale=2.5
)

show_multiple_images((back_person_img, coarse_res, refined_mask, final_res))
show_multiple_images(masks_refinement_steps)

# Save the front result as well to complete the set
filename = os.path.basename(back_person_path)
output_path = os.path.join(output_folder, filename)
final_res.save(output_path)
print(f"\nSaved front result to: {output_path}")

### Copy try-on results folder to Google Drive

## Download try-on results from Google Drive

In [ ]:
# unpack try-on result images
!mkdir -p /content/gaussian-splatting/input/vadim/try_on_res/input
!tar -xf /content/try_on_results/2_phase_vadim_2fps_the-north-face-retro-earth-short-sleeve-graphic-t-shirt-z-nadrukiem-white-dune-th322o0e8-a11.tar \
    --strip-components=1 \
    -C /content/gaussian-splatting/input/vadim/try_on_res/input

### Install GPU-accelerated COLMAP

### Create 3DGS model of vanila vadim

In [ ]:
!mkdir -p /content/gaussian-splatting/input/vadim/vanilla/input
!ffmpeg -i /content/gaussian-splatting/input/vadim/vadim.MOV -qscale: 1 -qmin 1 -vf fps=2 /content/gaussian-splatting/input/vadim/vanilla/input/%04d.jpg

In [ ]:
!mkdir -p /content/drive/MyDrive/weavella/3DGS_results/COLMAP_results
!tar -cf /content/drive/MyDrive/weavella/3DGS_results/COLMAP_results/vanilla_vadim_COLMAP_2fps_with_bg.tar \
    -C /content/gaussian-splatting/input/vadim vanilla

In [ ]:
!mkdir -p /content/gaussian-splatting/input/vadim/vanilla/
!gdown 1Y6fq0wR5Xs7_X55akFe3Ml_Px65W2tkz -O /content/gaussian-splatting/input/vadim/vanilla/vanilla_vadim_COLMAP_2fps_with_bg.tar

### Run 3DGS training on initial frames

In [ ]:
show_3DGS_loss_curve("/content/gaussian-splatting/output/vanilla_vadim_3DGS_with_bg")

In [ ]:
target_dir = Path("/content/gaussian-splatting/input/vadim/vanilla/images")
remove_background_from_all_in_dir(target_dir, automasker)

In [ ]:
# Start 3DGS training run and save it to a specific model folder (-m)
%cd /content/gaussian-splatting
!python train.py \
    -s /content/gaussian-splatting/input/vadim/vanilla/ \
    -m /content/gaussian-splatting/output/vanilla_vadim_3DGS_no_bg \
    --checkpoint_iterations 7000 30000 \
    --iterations 30000

In [ ]:
!tar -cf /content/drive/MyDrive/weavella/3DGS_results/just_tests_sandbox/vanilla_vadim_3DGS_2fps_no_bg.tar \
    -C /content/gaussian-splatting/output vanilla_vadim_3DGS_no_bg

### Optimize pre-trained 3DGS of vanilla Vadim on Using Try-On Images

Doesn't actually produce better results

In [ ]:
target_dir = Path("/content/gaussian-splatting/input/vadim/try_on_res/images")
remove_background_from_all_in_dir(target_dir, automasker)

In [ ]:
!python train.py \
    -s /content/gaussian-splatting/input/vadim/try_on_res/ \
    -m /content/gaussian-splatting/output/vanilla_optimized_try_on_vadim_3DGS \
    --start_checkpoint /content/gaussian-splatting/output/vanilla_vadim_3DGS_no_bg/chkpnt30000.pth \
    --checkpoint_iterations 37000 60000 \
    --iterations 60000

In [ ]:
!tar -cf /content/drive/MyDrive/weavella/3DGS_results/just_tests_sandbox/vanilla_optimized_try_on_vadim_3DGS_2fps_no_bg_v2.tar \
    -C /content/gaussian-splatting/output vanilla_optimized_try_on_vadim_3DGS

#### Create COLMAP from try-on images

In [ ]:
target_dir = Path("/content/gaussian-splatting/input/vadim/try_on_res/images")
remove_background_from_all_in_dir(target_dir, automasker)

In [ ]:
show_3DGS_loss_curve("/content/gaussian-splatting/output/vadim_try_on_3DGS_with_COLMAP_from_try_on_imgs")

In [ ]:
# Copy the specific training output folder to Google Drive
!tar -cf /content/drive/MyDrive/weavella/3DGS_results/just_tests_sandbox/vadim_try_on_2_phase_2fps_with_COLMAP_from_try_on_imgs_3DSG_30k_iter.tar \
    -C /content/gaussian-splatting/output vadim_try_on_3DGS_with_COLMAP_from_try_on_imgs

In [ ]:
# copy COLMAP created from vanilla frames, since we would have the same camera positions and init points
# -n flag to not replace our try-on images
!cp -rn /content/gaussian-splatting/input/vadim/vanilla/* /content/gaussian-splatting/input/vadim/try_on_res/
# -f flag to force replace/overwrite with our try-on images
!cp -rf /content/gaussian-splatting/input/vadim/try_on_res/input/* /content/gaussian-splatting/input/vadim/try_on_res/images/

In [ ]:
# Start 3DGS training run and save it to a specific model folder (-m)
%cd /content/gaussian-splatting
!python train.py \
    -s /content/gaussian-splatting/input/vadim/try_on_res/ \
    -m /content/gaussian-splatting/output/vadim_try_on_3DGS \
    --checkpoint_iterations 7000 30000 \
    --iterations 30000

In [ ]:
# --- RESUME 3DGS LATER ---
# Fix PyTorch weights_only strict loading issue in train.py
!sed -i 's/torch.load(checkpoint)/torch.load(checkpoint, weights_only=False)/g' /content/gaussian-splatting/train.py

# If the colab disconnects or you want to train further (e.g., to 40k), you can run:
!python train.py \
    -s /content/gaussian-splatting/input/vadim/try_on_res/ \
    -m /content/gaussian-splatting/output/vadim_try_on_3DGS \
    --start_checkpoint /content/gaussian-splatting/output/vadim_try_on_3DGS/chkpnt30000.pth \
    --checkpoint_iterations 100000 \
    --iterations 100000

# Freeze Pip versions

# Expert LoRAs Training

#### Determine body rotation angles from COLMAP

In [ ]:
# ---------------------------------------------------------------------------
# 1. COLMAP Parsing Utilities
# ---------------------------------------------------------------------------
def qvec2rotmat(qvec):
    return np.array([
        [1 - 2 * qvec[2]**2 - 2 * qvec[3]**2,
         2 * qvec[1] * qvec[2] - 2 * qvec[0] * qvec[3],
         2 * qvec[3] * qvec[1] + 2 * qvec[0] * qvec[2]],
        [2 * qvec[1] * qvec[2] + 2 * qvec[0] * qvec[3],
         1 - 2 * qvec[1]**2 - 2 * qvec[3]**2,
         2 * qvec[2] * qvec[3] - 2 * qvec[0] * qvec[1]],
        [2 * qvec[3] * qvec[1] - 2 * qvec[0] * qvec[2],
         2 * qvec[2] * qvec[3] + 2 * qvec[0] * qvec[1],
         1 - 2 * qvec[1]**2 - 2 * qvec[2]**2]
    ])

def read_images_binary(path_to_model_file):
    """
    Reads the COLMAP images.bin file.
    Returns a dict mapping image_id to image info including name and Q, T.
    """
    images = {}
    with open(path_to_model_file, "rb") as fid:
        num_reg_images = struct.unpack("<Q", fid.read(8))[0]
        for _ in range(num_reg_images):
            binary_image_properties = struct.unpack("<idddddddi", fid.read(64))
            image_id = binary_image_properties[0]
            qvec = np.array(binary_image_properties[1:5])
            tvec = np.array(binary_image_properties[5:8])
            camera_id = binary_image_properties[8]
            image_name = ""
            current_char = struct.unpack("<c", fid.read(1))[0]
            while current_char != b"\x00":   # look for the null \0
                image_name += current_char.decode("utf-8")
                current_char = struct.unpack("<c", fid.read(1))[0]
            num_points2D = struct.unpack("<Q", fid.read(8))[0]
            # Skip 2D points (x, y, point3D_id) = 8 bytes + 8 bytes + 8 bytes = 24 bytes
            fid.seek(num_points2D * 24, 1)

            images[image_id] = {
                "name": image_name,
                "qvec": qvec,
                "tvec": tvec
            }
    return images

def calculate_azimuth(qvec, tvec, centroid=None):
    """
    Calculate the azimuthal angle (in degrees) of the camera relative to the origin.
    If centroid is provided, assumes the subject is centered at the given centroid.
    Camera center C = -R^T * T
    """
    R = qvec2rotmat(qvec)
    C = -np.dot(R.T, tvec)

    if centroid is not None:
        C = C - centroid

    # In COLMAP, standard coordinate system often has Y pointing down, Z forward.
    # The XZ plane is usually the ground plane.
    # Let's compute angle using atan2(x, z). Adjust this mapping if your world orientation differs.
    x, y, z = C
    angle_rad = math.atan2(x, z)
    angle_deg = math.degrees(angle_rad)

    # Normalize to 0-360
    if angle_deg < 0:
        angle_deg += 360
    return angle_deg

# ---------------------------------------------------------------------------
# 2. Dataset Partitioning
# ---------------------------------------------------------------------------

def get_colmap_angles(colmap_images_bin):
    """
    Extracts images from COLMAP and calculates their raw azimuth angles.
    """
    images_info = read_images_binary(colmap_images_bin)

    # Calculate centroid of all cameras
    centers = []
    for info in images_info.values():
        R = qvec2rotmat(info["qvec"])
        C = -np.dot(R.T, info["tvec"])
        centers.append(C)
    centroid = np.mean(centers, axis=0)
    print(f"Calculated camera centroid (new origin): {centroid}")

    # Calculate raw angles
    raw_angles = {}
    for img_id, info in images_info.items():
        raw_angles[info["name"]] = calculate_azimuth(info["qvec"], info["tvec"], centroid)

    return raw_angles

def shift_angles(raw_angles, anchor_image_name=None):
    """
    Shifts all angles so that the anchor_image_name is at 0 degrees.
    """
    shift_angle = 0
    if anchor_image_name and anchor_image_name in raw_angles:
        shift_angle = raw_angles[anchor_image_name]
        print(f"Anchoring front to {anchor_image_name} (Raw Angle: {shift_angle:.2f} deg). Shifting all angles by -{shift_angle:.2f} deg.")

    shifted_angles = {}
    for name, angle in raw_angles.items():
        shifted_angles[name] = (angle - shift_angle) % 360

    return shifted_angles

def create_partitions(shifted_angles, source_image_dir, output_dir):
    """
    Partitions images into Front, Side, and Back folders based on shifted azimuths.
    """
    partitions = {
        "front": [],
        "side": [],
        "back": []
    }

    os.makedirs(os.path.join(output_dir, "front"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "side"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "back"), exist_ok=True)

    for name, angle in shifted_angles.items():
        # Splatting the Cat partitions:
        # Front: -60 to 60 (or 300 to 360 and 0 to 60)
        # Back: 120 to 240
        # Side: 60 to 120, and 240 to 300

        if angle <= 60 or angle >= 300:
            category = "front"
        elif 120 <= angle <= 240:
            category = "back"
        else:
            category = "side"

        partitions[category].append(name)

        # Copy file
        src_path = os.path.join(source_image_dir, name)
        dst_path = os.path.join(output_dir, category, name)
        if os.path.exists(src_path):
            shutil.copy2(src_path, dst_path)

    print(f"\nDataset partitioned: Front({len(partitions['front'])}), Side({len(partitions['side'])}), Back({len(partitions['back'])})")
    for name in sorted(shifted_angles.keys()):
        print(f"Image: {name:<12} | Angle: {shifted_angles[name]:>6.2f} deg")
    return partitions


In [ ]:
import os, random
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from PIL.ImageOps import exif_transpose
from tqdm.auto import tqdm
import torchvision.transforms as T
from diffusers import StableDiffusionInpaintPipeline, DDPMScheduler, AutoencoderKL
from diffusers.optimization import get_scheduler
from peft import LoraConfig, get_peft_model
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator

def make_mask(resolution, times=30):
    """RealFill-style mask (many small boxes). 1 = inpaint region, 0 = keep (context)."""
    mask = torch.ones(1, resolution, resolution)
    times = np.random.randint(1, times)
    min_size, max_size, margin = np.array([0.03, 0.25, 0.01]) * resolution
    max_size = min(max_size, resolution - margin * 2)
    for _ in range(times):
        w = np.random.randint(int(min_size), int(max_size))
        h = np.random.randint(int(min_size), int(max_size))
        x = np.random.randint(int(margin), resolution - int(margin) - w + 1)
        y = np.random.randint(int(margin), resolution - int(margin) - h + 1)
        mask[:, y:y + h, x:x + w] = 0
    if random.random() < 0.5:
        mask = 1 - mask
    return mask

def train_expert_lora(image_folder, output_dir, view_name,
                      rank=8, alpha=16, dropout=0.1, steps=1000,
                      batch_size=4, lr_unet=2e-4, lr_text=4e-5,
                      lr_warmup=100, resolution=512,
                      train_text_encoder=False,            # GS-VTON trains it; see note below
                      model_id="runwayml/stable-diffusion-inpainting"):  # MUST match SDSLoss base
    print(f"\n--- LoRA training: {view_name.upper()} (rank={rank}, steps={steps}) ---")
    accelerator = Accelerator(mixed_precision="fp16")
    device = accelerator.device

    tokenizer    = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
    text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder").to(device)
    vae          = AutoencoderKL.from_pretrained(model_id, subfolder="vae").to(device)
    unet         = StableDiffusionInpaintPipeline.from_pretrained(model_id, subfolder="unet").unet.to(device)
    scheduler    = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")
    assert scheduler.config.prediction_type == "epsilon"

    vae.requires_grad_(False)
    unet.requires_grad_(False)
    text_encoder.requires_grad_(False)

    # UNet LoRA (q/k/v/out)
    unet = get_peft_model(unet, LoraConfig(
        r=rank, lora_alpha=alpha, lora_dropout=dropout,
        target_modules=["to_q", "to_k", "to_v", "to_out.0"]))
    unet.train()

    param_groups = [{"params": [p for p in unet.parameters() if p.requires_grad], "lr": lr_unet}]
    if train_text_encoder:
        # GS-VTON / RealFill also adapts the text encoder. If you enable this, SDSLoss must
        # additionally load + switch the matching "lora_expert_<view>_text" adapters per view.
        text_encoder = get_peft_model(text_encoder, LoraConfig(
            r=rank, lora_alpha=alpha, lora_dropout=dropout,
            target_modules=["k_proj", "q_proj", "v_proj", "out_proj"]))
        text_encoder.train()
        param_groups.append({"params": [p for p in text_encoder.parameters() if p.requires_grad], "lr": lr_text})

    optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-2)
    lr_sched  = get_scheduler("constant_with_warmup", optimizer=optimizer,
                              num_warmup_steps=lr_warmup, num_training_steps=steps)

    # Load training images once
    paths = [os.path.join(image_folder, f) for f in os.listdir(image_folder)
             if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    if not paths:
        print(f"No images for {view_name}. Skipping.")
        return []
    print(f"{len(paths)} training images.")
    imgs = [exif_transpose(Image.open(p)).convert("RGB") for p in paths]

    aug = T.Compose([
        T.RandomResizedCrop(resolution, scale=(0.9, 1.0), ratio=(1.0, 1.0)),
        T.ToTensor(),                 # [0, 1]
        T.Normalize([0.5], [0.5]),    # -> [-1, 1]
    ])

    prompt = f"a photo of a {view_name} view person wearing a garment"

    unet, text_encoder, optimizer = accelerator.prepare(unet, text_encoder, optimizer)

    def encode_prompt(use_empty):
        txt = "" if use_empty else prompt
        ids = tokenizer(txt, padding="max_length", truncation=True,
                        max_length=tokenizer.model_max_length, return_tensors="pt").input_ids.to(device)
        if train_text_encoder:
            return text_encoder(ids)[0]
        with torch.no_grad():
            return text_encoder(ids)[0]

    losses = []
    for step in tqdm(range(steps), desc=f"Training {view_name}"):
        optimizer.zero_grad()
        idx = np.random.choice(len(imgs), size=batch_size, replace=True)
        pixels = torch.stack([aug(imgs[i]) for i in idx]).to(device)                          # [B,3,H,W] in [-1,1]
        masks  = torch.stack([make_mask(resolution) for _ in range(batch_size)]).to(device)   # [B,1,H,W]

        # --- CORRECT inpainting conditioning: apply mask in PIXEL space, THEN encode ---
        masked_pixels = pixels * (masks < 0.5)
        with torch.no_grad():
            latents    = vae.encode(pixels).latent_dist.sample()        * vae.config.scaling_factor
            masked_lat = vae.encode(masked_pixels).latent_dist.sample() * vae.config.scaling_factor
        masks_lat = F.interpolate(masks, size=latents.shape[2:])          # mask -> latent resolution

        noise = torch.randn_like(latents)
        t = torch.randint(0, scheduler.config.num_train_timesteps, (batch_size,), device=device).long()
        noisy = scheduler.add_noise(latents, noise, t)

        emb = encode_prompt(use_empty=(random.random() < 0.1))           # 10% CFG (empty-prompt) dropout
        if emb.shape[0] == 1:
            emb = emb.repeat(batch_size, 1, 1)

        model_in = torch.cat([noisy, masks_lat, masked_lat], dim=1)      # 4 + 1 + 4 = 9 channels
        pred = unet(model_in, t, encoder_hidden_states=emb).sample

        loss = F.mse_loss(pred.float(), noise.float())                   # uniform weighting (GS-VTON style)
        accelerator.backward(loss)
        accelerator.clip_grad_norm_(unet.parameters(), 1.0)
        optimizer.step(); lr_sched.step()
        losses.append(loss.item())

    # Save adapters. UNet dir name kept identical so the existing SDSLoss loader still works.
    os.makedirs(output_dir, exist_ok=True)
    accelerator.unwrap_model(unet).save_pretrained(os.path.join(output_dir, f"lora_expert_{view_name}"))
    if train_text_encoder:
        accelerator.unwrap_model(text_encoder).save_pretrained(os.path.join(output_dir, f"lora_expert_{view_name}_text"))
    print(f"Saved {view_name} expert to {output_dir}/lora_expert_{view_name}")
    return losses


In [ ]:
# 2. Partition dataset while anchoring the new zero degree mark to the true front
COLMAP_BIN_PATH = "/content/gaussian-splatting/input/vadim/try_on_res/sparse/0/images.bin"
SOURCE_IMAGES = "/content/gaussian-splatting/input/vadim/try_on_res/images"
PARTITION_DIR = "/content/dataset_partitioned"

raw_angles = get_colmap_angles(COLMAP_BIN_PATH)
shifted_angles = shift_angles(raw_angles, anchor_image_name=Path(front_person_path).name)
partitions = create_partitions(shifted_angles, SOURCE_IMAGES, PARTITION_DIR)

In [ ]:
for view in ["front", "side", "back"]:
    folder = os.path.join(PARTITION_DIR, view)
    files = sorted(os.listdir(folder))[:4]
    print(f"\n{view.upper()}  ({len(os.listdir(folder))} imgs): {files}")
    if files:
        fig, axes = plt.subplots(1, len(files), figsize=(4 * len(files), 4))
        for ax, f in zip(np.atleast_1d(axes), files):
            ax.imshow(Image.open(os.path.join(folder, f)))
            ax.set_title(f)
            ax.axis('off')
        plt.show()

In [ ]:
LORA_OUTPUT_DIR = "/content/expert_loras"

all_losses = {}

# Train the 3 view-experts with GS-VTON / RealFill mechanics (rank 8, ~1000 steps).
# IMPORTANT: model_id must match the base used by SDSLoss below.
for view in ["front", "side", "back"]:
    view_folder = os.path.join(PARTITION_DIR, view)
    if os.path.exists(view_folder) and os.listdir(view_folder):
        all_losses[view] = train_expert_lora(
            view_folder, LORA_OUTPUT_DIR, view,
            rank=8, alpha=16, dropout=0.1, steps=1000, batch_size=4,
            train_text_encoder=False,
            model_id="runwayml/stable-diffusion-inpainting")
    else:
        print(f"Skipping {view}: empty or missing folder {view_folder}.")

In [ ]:
# save trained expert LoRAs to Google Drive
!mkdir -p /content/drive/MyDrive/weavella/LoRA_results/expert_loras
!tar -cf /content/drive/MyDrive/weavella/LoRA_results/expert_loras/expert_loras_sd15_try_on_v1.tar -C /content/expert_loras/ .

### Train view-expert LoRAs using SD 2.0
Reusing the `train_expert_lora` function by specifying the SD 2.0 inpainting model.

In [ ]:
plt.figure(figsize=(10, 6))
for view, losses in all_losses_sd2.items():
    if losses:
        # Smooth the curve for better readability
        smoothed_losses = [np.mean(losses[max(0, i-50):i+1]) for i in range(len(losses))]
        plt.plot(smoothed_losses, label=f"{view} Expert")

plt.xlabel("Training Steps")
plt.ylabel("Loss (MSE)")
plt.title("LoRA Training Loss Curves")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Download trained LoRAs
!mkdir -p /content/expert_loras_sd2
!gdown 1WWE0im9oGZDjLQqU3bVvtOVsi6Pq27ts -O /content/expert_loras_sd2/expert_loras_sd2_try_on_v1.tar
!tar -xf /content/expert_loras_sd2/expert_loras_sd2_try_on_v1.tar -C /content/expert_loras_sd2

In [ ]:
import os
import torch
from diffusers import StableDiffusionInpaintPipeline
import matplotlib.pyplot as plt
from PIL import Image
import PIL.ImageDraw as ImageDraw
from peft import PeftModel

view = "back"                                       # <-- the expert you want to test
model_id = "runwayml/stable-diffusion-inpainting"   # keep == the base you trained with
# model_id = "sd2-community/stable-diffusion-2-inpainting"

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    model_id, torch_dtype=torch.float16, safety_checker=None).to("cuda")
pipe.unet = PeftModel.from_pretrained(pipe.unet, f"/content/expert_loras/lora_expert_{view}")

# Use a real image from that view's partition
view_dir = os.path.join(PARTITION_DIR, view)
test_path = os.path.join(view_dir, sorted(os.listdir(view_dir))[0])
test_img = Image.open(test_path).convert("RGB").resize((512, 512))

# Partial mask over the torso (where the garment lives). 255 = inpaint.
mask_image = Image.new("L", (512, 512), 0)
ImageDraw.Draw(mask_image).rectangle([150, 180, 360, 420], fill=255)

print(f"Testing '{view}' expert by reconstructing a real {view} view...")
result = pipe(
    prompt=f"a photo of a {view} view person wearing a garment",
    image=test_img,
    mask_image=mask_image,
    num_inference_steps=50,
    guidance_scale=4.0,
    generator=torch.Generator("cuda").manual_seed(0)
).images[0]

show_multiple_images([test_img, mask_image, result])

## Score Distillation Sampling (SDS) Loss

To optimize 3D Gaussians using our 2D diffusion model, we use Score Distillation Sampling. We will load all our view-expert LoRAs into the UNet simultaneously using `PeftModel`'s multi-adapter support. Depending on the camera view being rendered, we switch to the corresponding active adapter.

### Integration into 3DGS `train.py`

In [ ]:
%%writefile /content/gaussian-splatting/train_sds.py
import os
import torch
import math
import numpy as np
from random import randint
from utils.loss_utils import l1_loss, ssim
from gaussian_renderer import render, network_gui
import sys
from scene import Scene, GaussianModel
from utils.general_utils import safe_state
import uuid
from tqdm import tqdm
from utils.image_utils import psnr
from argparse import ArgumentParser, Namespace
from arguments import ModelParams, PipelineParams, OptimizationParams
try:
    from torch.utils.tensorboard import SummaryWriter
    TENSORBOARD_FOUND = True
except ImportError:
    TENSORBOARD_FOUND = False

from sds_loss import SDSLoss

def determine_view_from_camera(cam, centroid, shift_angle):
    C = cam.camera_center.detach().cpu().numpy()
    # Shift relative to dataset centroid
    C = C - centroid
    x, y, z = C[0], C[1], C[2]

    angle_deg = math.degrees(math.atan2(x, z))
    if angle_deg < 0:
        angle_deg += 360

    # Apply anchoring shift
    angle_deg = (angle_deg - shift_angle) % 360

    if angle_deg <= 60 or angle_deg >= 300:
        return "front"
    elif 120 <= angle_deg <= 240:
        return "back"
    else:
        return "side"

def training(dataset, opt, pipe, testing_iterations, saving_iterations, checkpoint_iterations, checkpoint, debug_from, front_image_name):
    first_iter = 0
    tb_writer = prepare_output_and_logger(dataset)
    gaussians = GaussianModel(dataset.sh_degree)
    scene = Scene(dataset, gaussians)
    gaussians.training_setup(opt)
    if checkpoint:
        (model_params, first_iter) = torch.load(checkpoint, weights_only=False)
        gaussians.restore(model_params, opt)

    bg_color = [1, 1, 1] if dataset.white_background else [0, 0, 0]
    background = torch.tensor(bg_color, dtype=torch.float32, device="cuda")

    # --- Dataset Centroid and Shift Angle Calculation ---
    train_cameras = scene.getTrainCameras()
    centers = [cam.camera_center.detach().cpu().numpy() for cam in train_cameras]
    centroid = np.mean(centers, axis=0)

    shift_angle = 0.0
    for cam in train_cameras:
        if cam.image_name == front_image_name:
            C = cam.camera_center.detach().cpu().numpy() - centroid
            shift_angle = math.degrees(math.atan2(C[0], C[2]))
            if shift_angle < 0:
                shift_angle += 360
            print(f"Found anchor front camera '{front_image_name}' at raw angle {shift_angle:.2f} deg.")
            break

    # Initialize SDS Loss Module
    print("Initializing SDS Loss Module...")
    sds_loss_module = SDSLoss(lora_dir="/content/expert_loras", device="cuda")

    iter_start = torch.cuda.Event(enable_timing = True)
    iter_end = torch.cuda.Event(enable_timing = True)

    viewpoint_stack = None
    ema_loss_for_log = 0.0
    progress_bar = tqdm(range(first_iter, opt.iterations), desc="Training progress")
    first_iter += 1
    for iteration in range(first_iter, opt.iterations + 1):
        if network_gui.conn == None:
            network_gui.try_connect()
        while network_gui.conn != None:
            try:
                net_image_bytes = None
                custom_cam, do_training, pipe.convert_SHs_python, pipe.compute_cov3D_python, keep_alive, scaling_modifer = network_gui.receive()
                if custom_cam != None:
                    net_image = render(custom_cam, gaussians, pipe, background, scaling_modifer)["render"]
                    net_image_bytes = memoryview((torch.clamp(net_image, min=0, max=1.0) * 255).byte().permute(1, 2, 0).contiguous().cpu().numpy())
                network_gui.send(net_image_bytes, dataset.source_path)
                if do_training and ((iteration < int(opt.iterations)) or not keep_alive):
                    break
            except Exception as e:
                network_gui.conn = None

        iter_start.record()

        gaussians.update_learning_rate(iteration)

        if iteration % 1000 == 0:
            gaussians.oneupSHdegree()

        if not viewpoint_stack:
            viewpoint_stack = scene.getTrainCameras().copy()
        viewpoint_cam = viewpoint_stack.pop(randint(0, len(viewpoint_stack)-1))

        if (iteration - 1) == debug_from:
            pipe.debug = True
        render_pkg = render(viewpoint_cam, gaussians, pipe, background)
        image, viewspace_point_tensor, visibility_filter, radii = render_pkg["render"], render_pkg["viewspace_points"], render_pkg["visibility_filter"], render_pkg["radii"]

        gt_image = viewpoint_cam.original_image.cuda()
        Ll1 = l1_loss(image, gt_image)
        standard_loss = (1.0 - opt.lambda_dssim) * Ll1 + opt.lambda_dssim * (1.0 - ssim(image, gt_image))

        # Apply SDS selectively every 5 iterations to save massive compute time
        if iteration % 5 == 0:
            view_type = determine_view_from_camera(viewpoint_cam, centroid, shift_angle)
            loss_sds = sds_loss_module.get_sds_loss(image, view_type=view_type)
            # Weighting 0.1 for standard L1 to keep background stable, and SDS for the generative updates
            loss = loss_sds * 0.1 + standard_loss * 0.9
        else:
            loss = standard_loss

        loss.backward()

        iter_end.record()

        with torch.no_grad():
            ema_loss_for_log = 0.4 * loss.item() + 0.6 * ema_loss_for_log
            if iteration % 10 == 0:
                progress_bar.set_postfix({"Loss": f"{ema_loss_for_log:.{7}f}"})
                progress_bar.update(10)
            if iteration == opt.iterations:
                progress_bar.close()

            training_report(tb_writer, iteration, Ll1, loss, l1_loss, iter_start.elapsed_time(iter_end), testing_iterations, scene, render, (pipe, background))
            if (iteration in saving_iterations):
                print("\n[ITER {}] Saving Gaussians".format(iteration))
                scene.save(iteration)

            if iteration < opt.densify_until_iter:
                gaussians.max_radii2D[visibility_filter] = torch.max(gaussians.max_radii2D[visibility_filter], radii[visibility_filter])
                gaussians.add_densification_stats(viewspace_point_tensor, visibility_filter)

                if iteration > opt.densify_from_iter and iteration % opt.densification_interval == 0:
                    size_threshold = 20 if iteration > opt.opacity_reset_interval else None
                    gaussians.densify_and_prune(opt.densify_grad_threshold, 0.005, scene.cameras_extent, size_threshold)

                if iteration % opt.opacity_reset_interval == 0 or (dataset.white_background and iteration == opt.densify_from_iter):
                    gaussians.reset_opacity()

            if iteration < opt.iterations:
                gaussians.optimizer.step()
                gaussians.optimizer.zero_grad(set_to_none = True)

            if (iteration in checkpoint_iterations):
                print("\n[ITER {}] Saving Checkpoint".format(iteration))
                torch.save((gaussians.capture(), iteration), scene.model_path + "/chkpnt" + str(iteration) + ".pth")

def prepare_output_and_logger(args):
    if not args.model_path:
        if os.getenv('OAR_JOB_ID'):
            unique_str=os.getenv('OAR_JOB_ID')
        else:
            unique_str = str(uuid.uuid4())
        args.model_path = os.path.join("./output/", unique_str[0:10])

    print("Output folder: ".format(args.model_path))
    os.makedirs(args.model_path, exist_ok = True)
    with open(os.path.join(args.model_path, "cfg_args"), 'w') as cfg_log_f:
        cfg_log_f.write(str(Namespace(**vars(args))))

    tb_writer = None
    if TENSORBOARD_FOUND:
        tb_writer = SummaryWriter(args.model_path)
    else:
        print("Tensorboard not available: not logging progress")
    return tb_writer

def training_report(tb_writer, iteration, Ll1, loss, l1_loss, elapsed, testing_iterations, scene : Scene, renderFunc, renderArgs):
    if tb_writer:
        tb_writer.add_scalar('train_loss_patches/l1_loss', Ll1.item(), iteration)
        tb_writer.add_scalar('train_loss_patches/total_loss', loss.item(), iteration)
        tb_writer.add_scalar('iter_time', elapsed, iteration)

    if iteration in testing_iterations:
        torch.cuda.empty_cache()
        validation_configs = ({'name': 'test', 'cameras' : scene.getTestCameras()},
                              {'name': 'train', 'cameras' : [scene.getTrainCameras()[idx % len(scene.getTrainCameras())] for idx in range(5, 30, 5)]})

        for config in validation_configs:
            if config['cameras'] and len(config['cameras']) > 0:
                l1_test = 0.0
                psnr_test = 0.0
                for idx, viewpoint in enumerate(config['cameras']):
                    image = torch.clamp(renderFunc(viewpoint, scene.gaussians, *renderArgs)["render"], 0.0, 1.0)
                    gt_image = torch.clamp(viewpoint.original_image.to("cuda"), 0.0, 1.0)
                    if tb_writer and (idx < 5):
                        tb_writer.add_images(config['name'] + "_view_{}/render".format(viewpoint.image_name), image[None], global_step=iteration)
                        if iteration == testing_iterations[0]:
                            tb_writer.add_images(config['name'] + "_view_{}/ground_truth".format(viewpoint.image_name), gt_image[None], global_step=iteration)
                    l1_test += l1_loss(image, gt_image).mean().double()
                    psnr_test += psnr(image, gt_image).mean().double()
                psnr_test /= len(config['cameras'])
                l1_test /= len(config['cameras'])
                print("\n[ITER {}] Evaluating {}: L1 {} PSNR {}".format(iteration, config['name'], l1_test, psnr_test))
                if tb_writer:
                    tb_writer.add_scalar(config['name'] + '/loss_viewpoint - l1_loss', l1_test, iteration)
                    tb_writer.add_scalar(config['name'] + '/loss_viewpoint - psnr', psnr_test, iteration)

        if tb_writer:
            tb_writer.add_histogram("scene/opacity_histogram", scene.gaussians.get_opacity, iteration)
            tb_writer.add_scalar('total_points', scene.gaussians.get_xyz.shape[0], iteration)
        torch.cuda.empty_cache()

if __name__ == "__main__":
    parser = ArgumentParser(description="Training script parameters")
    lp = ModelParams(parser)
    op = OptimizationParams(parser)
    pp = PipelineParams(parser)
    parser.add_argument('--ip', type=str, default="127.0.0.1")
    parser.add_argument('--port', type=int, default=6009)
    parser.add_argument('--debug_from', type=int, default=-1)
    parser.add_argument('--detect_anomaly', action='store_true', default=False)
    parser.add_argument("--test_iterations", nargs="+", type=int, default=[7_000, 30_000])
    parser.add_argument("--save_iterations", nargs="+", type=int, default=[7_000, 30_000])
    parser.add_argument("--quiet", action="store_true")
    parser.add_argument("--checkpoint_iterations", nargs="+", type=int, default=[])
    parser.add_argument("--start_checkpoint", type=str, default = None)
    parser.add_argument("--front_image_name", type=str, default="0011", help="The name of the true front image to use as 0-degree anchor")
    args = parser.parse_args(sys.argv[1:])
    args.save_iterations.append(args.iterations)

    print("Optimizing " + args.model_path)
    safe_state(args.quiet)
    network_gui.init(args.ip, args.port)
    torch.autograd.set_detect_anomaly(args.detect_anomaly)
    training(lp.extract(args), op.extract(args), pp.extract(args), args.test_iterations, args.save_iterations, args.checkpoint_iterations, args.start_checkpoint, args.debug_from, args.front_image_name)
    print("\nTraining complete.")


In [ ]:
%cd /content/gaussian-splatting

!python train_sds.py \
    -s /content/gaussian-splatting/input/vadim/try_on_res/ \
    -m /content/gaussian-splatting/output/vadim_try_on_3DGS_sds \
    --front_image_name "0011.jpg" \
    --iterations 35000 \
    --checkpoint_iterations 35000

In [ ]:
import torch
import numpy as np
from tqdm.auto import tqdm
import sys
import os

# Ensure we can import from the gaussian-splatting directory
if "/content/gaussian-splatting" not in sys.path:
    sys.path.append("/content/gaussian-splatting")

from scene import Scene, GaussianModel
from gaussian_renderer import render
from argparse import ArgumentParser
from arguments import ModelParams, PipelineParams, OptimizationParams

def evaluate_scene_masked_psnr(model_path, source_path):
    # 1. Setup parameters
    parser = ArgumentParser(description="Testing script parameters")
    model = ModelParams(parser)
    pipeline = PipelineParams(parser)

    # Mock arguments for loading
    args = parser.parse_args([])
    args.model_path = model_path
    args.source_path = source_path
    args.sh_degree = 3
    args.white_background = False
    args.resolution = -1
    args.data_device = "cuda"
    args.eval = False

    dataset = model.extract(args)
    pipe = pipeline.extract(args)

    # 2. Load Model
    print(f"Loading 3DGS model from: {model_path}")
    gaussians = GaussianModel(dataset.sh_degree)
    scene = Scene(dataset, gaussians, load_iteration=-1, shuffle=False) # Load the latest checkpoint

    bg_color = [1, 1, 1] if dataset.white_background else [0, 0, 0]
    background = torch.tensor(bg_color, dtype=torch.float32, device="cuda")

    # Use train cameras (or test cameras if you passed --eval during training)
    cameras = scene.getTrainCameras()

    total_psnr = 0.0
    valid_frames = 0

    print(f"Evaluating {len(cameras)} cameras for Masked PSNR...")
    with torch.no_grad():
        for cam in tqdm(cameras):
            # Render the viewpoint
            render_pkg = render(cam, gaussians, pipe, background)
            rendered_image = render_pkg["render"]
            gt_image = cam.original_image.cuda()

            # Create a binary mask from the GT image (assuming the background was removed to pure black)
            # We sum across color channels and check where it's > 0 (or > a small epsilon like 0.05)
            mask = (gt_image.sum(dim=0) > 0.05).float()

            # Calculate masked PSNR using your custom function
            psnr_val = calculate_masked_psnr(rendered_image, gt_image, mask)

            if psnr_val != float('inf'):
                total_psnr += psnr_val
                valid_frames += 1

    avg_psnr = total_psnr / valid_frames if valid_frames > 0 else 0
    print(f"\n---> Average Masked PSNR across {valid_frames} views: {avg_psnr:.4f} dB")
    return avg_psnr

# --- Example Usage ---
# Let's test the PSNR of the SDS fine-tuned try-on model:
model_dir = "/content/gaussian-splatting/output/vadim_try_on_3DGS_sds"
source_dir = "/content/gaussian-splatting/input/vadim/try_on_res/"

evaluate_scene_masked_psnr(model_dir, source_dir)